In [ ]:
import warnings
warnings.filterwarnings("ignore")

import wikipedia

from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from sentence_transformers import CrossEncoder


# ---------------------------------------------------------
# 1. Load document
# ---------------------------------------------------------

wikipedia.set_user_agent("my-rag-project/1.0")

loader = WikipediaLoader(
    query="Retrieval-augmented generation",
    load_max_docs=1,
    doc_content_chars_max=1000000
)

docs = loader.load()


# ---------------------------------------------------------
# 2. Split document into chunks
# ---------------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

# print("Number of chunks:", len(chunks))


# ---------------------------------------------------------
# 3. Bi-encoder
# ---------------------------------------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)


# ---------------------------------------------------------
# 4. Create FAISS vector store
# ---------------------------------------------------------

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)


# ---------------------------------------------------------
# 5. User query
# ---------------------------------------------------------

query = "How does RAG retrieve relevant information?"


# ---------------------------------------------------------
# 6. BI-ENCODER RETRIEVAL
# ---------------------------------------------------------

# Retrieve more candidates than we ultimately need.
# For example, retrieve top 10 candidates.
retrieved_docs = vector_store.similarity_search(
    query,
    k=10
)

# print("\n========== BI-ENCODER RETRIEVAL ==========")

# for i, doc in enumerate(retrieved_docs):
#     print(f"\nCandidate {i + 1}")
#     print(doc.page_content[:500])


# ---------------------------------------------------------
# 7. Cross-encoder
# ---------------------------------------------------------

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


# ---------------------------------------------------------
# 8. Create (query, document) pairs
# ---------------------------------------------------------

pairs = [
    (query, doc.page_content)
    for doc in retrieved_docs
]


# ---------------------------------------------------------
# 9. CROSS-ENCODER RERANKING
# ---------------------------------------------------------

scores = cross_encoder.predict(pairs)


# ---------------------------------------------------------
# 10. Attach scores to documents
# ---------------------------------------------------------

scored_docs = list(zip(retrieved_docs, scores))


# Higher cross-encoder score = more relevant
scored_docs = sorted(
    scored_docs,
    key=lambda x: x[1],
    reverse=True
)


# ---------------------------------------------------------
# 11. Select final Top-K documents
# ---------------------------------------------------------

top_k = 5

reranked_docs = scored_docs[:top_k]


# ---------------------------------------------------------
# 12. Display final results
# ---------------------------------------------------------

# print("\n========== CROSS-ENCODER RERANKING ==========")

# for rank, (doc, score) in enumerate(reranked_docs, start=1):

#     print(f"\nRank {rank}")
#     print(f"Cross-encoder score: {score:.4f}")
#     print(doc.page_content[:1000])